# Figure 3: model performance

Model-performance figure using the final 553-sequence dataset, feature-block panel A, and Platt-calibrated LightGBM predictions for panels B-D.


In [9]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from adjustText import adjust_text
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, balanced_accuracy_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold

NOTEBOOK_DIR = Path.cwd().resolve()
CANDIDATES = [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]
REPO_ROOT = next(path for path in CANDIDATES if (path / "model_test").exists())

COMPARE_SUITE_DIR = REPO_ROOT / "model_test" / "runs" / "model_site" / "model_suite_final553_figure_model_performance"
FOCAL_SUITE_DIR = COMPARE_SUITE_DIR
PANEL_A_COMPARE_DIR = REPO_ROOT / "final_model" / "figure_inputs" / "figure1_compare_latest_winners"
FIGURE_SAVE_PATH = REPO_ROOT / "notebooks" / "figures"
FIGURE_SAVE_PATH.mkdir(parents=True, exist_ok=True)

print(REPO_ROOT)
print(FIGURE_SAVE_PATH)


/Users/ricardorivero/Documents/GitHub/HantaReassortment
/Users/ricardorivero/Documents/GitHub/HantaReassortment/notebooks/figures


In [10]:
SCHEME_LABELS = {
    "grouped_species": "Leave-one-species-out",
    "grouped_within_species_clade": "Leave-one-lineage-out",
    "repeated_stratified": "Repeated stratified CV",
}

FEATURE_SET_DISPLAY = {
    "species_only": "Species only",
    "ecology_only": "Ecology only",
    "molecular_only": "Molecular only",
    "ecology_molecular": "Ecology + molecular",
    "integrated_taxon_mechanism": "Integrated",
    "ecology_molecular_lineage_core": "Lineage core",
    "ecology_molecular_lineage_plus_context": "Lineage + context",
    "ecology_molecular_lineage_plus_species_summaries": "Lineage + species summaries",
}

MODEL_NAME_DISPLAY = {
    "random_forest_shallow": "Random forest",
    "lightgbm": "LightGBM",
    "xgboost": "XGBoost",
    "catboost": "CatBoost",
    "gradient_boosting": "Gradient boosting",
    "penalized_logistic_l2": "L2 logistic",
    "penalized_logistic_elasticnet": "Elastic-net logistic",
}

SPECIES_COLORS = {
    "orthohantavirus_andesense": "#1b9e77",
    "orthohantavirus_dobravaense": "#d95f02",
    "orthohantavirus_hantanense": "#7570b3",
    "orthohantavirus_puumalaense": "#66a61e",
    "orthohantavirus_seoulense": "#e7298a",
    "orthohantavirus_sinnombreense": "#a6761d",
    "orthohantavirus_tulaense": "#1f78b4",
}

SPECIES_LABELS = {
    "orthohantavirus_andesense": "ANDV",
    "orthohantavirus_dobravaense": "DOBV",
    "orthohantavirus_hantanense": "HTNV",
    "orthohantavirus_puumalaense": "PUUV",
    "orthohantavirus_seoulense": "SEOV",
    "orthohantavirus_sinnombreense": "SNV",
    "orthohantavirus_tulaense": "TULV",
}

def set_style():
    mpl.rcParams.update(
        {
            "font.family": "Helvetica Neue",
            "font.size": 7,
            "axes.titlesize": 7.5,
            "axes.labelsize": 7,
            "xtick.labelsize": 6.5,
            "ytick.labelsize": 6.5,
            "legend.fontsize": 6.5,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "axes.facecolor": "white",
            "figure.facecolor": "white",
            "savefig.facecolor": "white",
            "axes.linewidth": 0.7,
            "xtick.major.width": 0.7,
            "ytick.major.width": 0.7,
            "xtick.major.size": 2.5,
            "ytick.major.size": 2.5,
            "lines.linewidth": 1.0,
        }
    )

def species_short(name):
    return SPECIES_LABELS.get(name, name)

def lighten_color(color, amount=0.45):
    rgb = np.array(mpl.colors.to_rgb(color))
    return tuple(rgb + (1.0 - rgb) * amount)

def panel_label(ax, label, x=-0.10, y=1.03):
    ax.text(x, y, label.lower(), transform=ax.transAxes, fontsize=10, fontweight="bold", va="top")

def pretty_model_name(name):
    if name in MODEL_NAME_DISPLAY:
        return MODEL_NAME_DISPLAY[name]
    return str(name).replace("_", " ").title()

set_style()


In [11]:
def make_figure3(compare_suite_dir=COMPARE_SUITE_DIR, repeat_suite_dir=FOCAL_SUITE_DIR, panel_a_dir=PANEL_A_COMPARE_DIR, outpath=None, focal_model="lightgbm", calibration="platt"):
    outpath = Path(outpath or FIGURE_SAVE_PATH / "figure3_model_performance.jpeg")
    outpath.parent.mkdir(parents=True, exist_ok=True)
    panel_a_dir = Path(panel_a_dir)
    if panel_a_dir.exists():
        fold_df = pd.read_csv(panel_a_dir / "cv_fold_metrics.tsv", sep="\t")
        summary_df = pd.read_csv(panel_a_dir / "summary.tsv", sep="\t")
    else:
        fold_df = pd.read_csv(compare_suite_dir / "cv_fold_metrics.tsv", sep="\t")
        summary_df = pd.read_csv(compare_suite_dir / "summary.tsv", sep="\t")
    repeat_pred_df = pd.read_csv(repeat_suite_dir / "repeated_stratified_cv_predictions.tsv", sep="\t")

    feature_set = "ecology_molecular_lineage_plus_context"
    feature_order = [
        "species_only",
        "ecology_only",
        "molecular_only",
        "ecology_molecular",
        "ecology_molecular_lineage_core",
        "ecology_molecular_lineage_plus_context",
    ]
    feature_colors = {
        "species_only": "#c7d7a1",
        "ecology_only": "#b7d7b0",
        "molecular_only": "#c7b7d9",
        "ecology_molecular": "#efc3a4",
        "ecology_molecular_lineage_core": "#f1d6a8",
        "ecology_molecular_lineage_plus_context": "#9fc5d5",
    }
    scheme_order = ["repeated_stratified", "grouped_within_species_clade", "grouped_species"]
    scheme_labels = {
        "repeated_stratified": "Repeated\nstratified",
        "grouped_within_species_clade": "Leave-one\nlineage-out",
        "grouped_species": "Leave-one\nspecies-out",
    }
    workhorse_by_scheme = {
        "repeated_stratified": "lightgbm",
        "grouped_within_species_clade": "xgboost",
        "grouped_species": "random_forest_shallow",
    }
    workhorse_labels = {
        "repeated_stratified": "LightGBM",
        "grouped_within_species_clade": "XGBoost",
        "grouped_species": "RF",
    }

    fold_df = fold_df[
        fold_df["cv_scheme"].isin(scheme_order)
        & fold_df["feature_set"].isin(feature_order)
        & fold_df.apply(lambda r: r["model"] == workhorse_by_scheme.get(r["cv_scheme"]), axis=1)
    ].copy()
    summary_df = summary_df[
        summary_df["cv_scheme"].isin(scheme_order)
        & summary_df["feature_set"].isin(feature_order)
        & summary_df.apply(lambda r: r["model"] == workhorse_by_scheme.get(r["cv_scheme"]), axis=1)
    ].copy()
    repeat_pred_df = repeat_pred_df[
        (repeat_pred_df["feature_set"] == feature_set)
        & (repeat_pred_df["model"] == focal_model)
    ].copy()
    if repeat_pred_df.empty:
        raise ValueError(f"No repeated-stratified predictions found for {feature_set} / {focal_model}")

    repeat_plot = (
        repeat_pred_df.groupby(["row_index", "tip_name", "species", "target"], as_index=False)["probability"]
        .mean()
        .reset_index(drop=True)
    )
    y_oof = repeat_plot["target"].to_numpy(dtype=int)
    raw_prob = repeat_plot["probability"].to_numpy(dtype=float)
    calibrated_prob = raw_prob.copy()
    calibration_label = "raw"
    if calibration == "platt":
        saved_calibration = compare_suite_dir / "repeated_stratified_lightgbm_platt_calibrated_oof_predictions.tsv"
        if focal_model == "lightgbm" and saved_calibration.exists():
            repeat_plot = pd.read_csv(saved_calibration, sep="\t").rename(
                columns={"calibrated_probability": "probability"}
            )
            raw_prob = repeat_plot["raw_probability"].to_numpy(dtype=float)
            calibrated_prob = repeat_plot["probability"].to_numpy(dtype=float)
            y_oof = repeat_plot["target"].to_numpy(dtype=int)
        else:
            eps = 1e-6
            logit_prob = np.log(np.clip(raw_prob, eps, 1 - eps) / (1 - np.clip(raw_prob, eps, 1 - eps))).reshape(-1, 1)
            calibrated_prob = np.full_like(raw_prob, np.nan, dtype=float)
            splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
            for train_idx, test_idx in splitter.split(logit_prob, y_oof):
                calibrator = LogisticRegression(solver="lbfgs")
                calibrator.fit(logit_prob[train_idx], y_oof[train_idx])
                calibrated_prob[test_idx] = calibrator.predict_proba(logit_prob[test_idx])[:, 1]
            repeat_plot["raw_probability"] = raw_prob
            repeat_plot["probability"] = calibrated_prob
        calibration_label = "Platt-calibrated"
    elif calibration not in {None, "raw"}:
        raise ValueError(f"Unsupported calibration mode: {calibration}")
    else:
        repeat_plot["raw_probability"] = raw_prob
        repeat_plot["probability"] = calibrated_prob
    repeat_plot = repeat_plot.sort_values("probability", ascending=False).reset_index(drop=True)
    prevalence = float(repeat_plot["target"].mean())
    n_total = len(repeat_plot)

    fig = plt.figure(figsize=(12.6, 7.5), constrained_layout=True)
    gs = fig.add_gridspec(2, 3, height_ratios=[1.05, 1.0], width_ratios=[1.18, 1.0, 1.02])
    axA = fig.add_subplot(gs[0, :])
    axB = fig.add_subplot(gs[1, 0])
    axC = fig.add_subplot(gs[1, 1])
    axD = fig.add_subplot(gs[1, 2])

    panel_label(axA, "a)")
    panel_label(axB, "b)", y=1.08)
    panel_label(axC, "c)", y=1.08)
    panel_label(axD, "d)", y=1.08)
    for ax in (axB, axC, axD):
        ax.set_box_aspect(0.95)

    positions = []
    labels = []
    centers = []
    cursor = 0.0
    for scheme in scheme_order:
        scheme_positions = []
        for fset in feature_order:
            pos = cursor
            positions.append(pos)
            labels.append(FEATURE_SET_DISPLAY.get(fset, fset).replace(" + ", " +\n").replace("Lineage ", "Lineage\n"))
            scheme_positions.append(pos)
            vals = fold_df[(fold_df["cv_scheme"] == scheme) & (fold_df["feature_set"] == fset)]["average_precision"].dropna().to_numpy()
            if len(vals):
                vp = axA.violinplot(vals, positions=[pos], widths=0.58, showextrema=False)
                for body in vp["bodies"]:
                    body.set_facecolor(feature_colors[fset])
                    body.set_edgecolor("none")
                    body.set_alpha(0.38)
                axA.boxplot(
                    vals,
                    positions=[pos],
                    widths=0.24,
                    patch_artist=True,
                    showfliers=False,
                    medianprops={"color": "#333333", "linewidth": 0.9},
                    whiskerprops={"color": "#555555", "linewidth": 0.7},
                    capprops={"color": "#555555", "linewidth": 0.7},
                    boxprops={"facecolor": "#fbfbfb", "edgecolor": "#555555", "linewidth": 0.7},
                )
                jitter = np.linspace(-0.075, 0.075, len(vals)) if len(vals) > 1 else np.array([0.0])
                axA.scatter(
                    np.full(len(vals), pos) + jitter,
                    vals,
                    s=11,
                    color=feature_colors[fset],
                    alpha=0.72,
                    edgecolor="white",
                    linewidth=0.25,
                    zorder=3,
                )
            stat = summary_df[(summary_df["cv_scheme"] == scheme) & (summary_df["feature_set"] == fset)]
            if not stat.empty:
                axA.scatter(
                    [pos],
                    [float(stat["oof_average_precision"].iloc[0])],
                    marker="D",
                    s=22,
                    color="#202020",
                    edgecolor="white",
                    linewidth=0.35,
                    zorder=4,
                )
            cursor += 0.72
        centers.append((np.mean(scheme_positions), f"{scheme_labels[scheme]}"))
        cursor += 0.88

    group_half_width = (len(feature_order) - 1) * 0.72 / 2 + 0.36
    for i, (center, label) in enumerate(centers):
        start = center - group_half_width
        end = center + group_half_width
        axA.axvspan(start, end, color="#fafaf8" if i % 2 == 0 else "#ffffff", zorder=0)
        axA.text(center, 0.985, label, transform=axA.get_xaxis_transform(), ha="center", va="top", fontsize=10, color="#4d4d4d")
    axA.axhline(prevalence, color="#555555", linestyle=(0, (4, 4)), linewidth=1.0)
    axA.text(max(positions) + 0.18, prevalence + 0.008, "Prevalence", fontsize=10, color="#555555")
    axA.set_xlim(min(positions) - 0.55, max(positions) + 0.7)
    axA.set_ylim(0.0, 1.0)
    axA.set_ylabel("Average precision", fontsize=10)
    axA.set_xticks(positions, labels)
    axA.tick_params(axis="x", labelsize=5.4, pad=2)

    x = np.arange(n_total)
    probs = repeat_plot["probability"].to_numpy()
    target = repeat_plot["target"].to_numpy()
    axB.fill_between(x, 0, probs, color="#fee8c8", alpha=0.78)
    axB.plot(x, probs, color="#2c3e50", linewidth=1.7)
    axB.axhline(prevalence, color="#555555", linestyle=(0, (4, 4)), linewidth=1.0)
    pos_idx = np.where(target == 1)[0]
    neg_idx = np.where(target == 0)[0]
    axB.scatter(pos_idx, np.full(pos_idx.shape, 1.01), marker="|", s=120, color="#bc4749", clip_on=False, transform=axB.get_xaxis_transform())
    neg_sub = neg_idx[:: max(1, len(neg_idx) // 120)]
    axB.scatter(neg_sub, np.full(len(neg_sub), -0.045), marker="|", s=45, color="#b8b8b8", clip_on=False)
    for i, species in enumerate(repeat_plot["species"]):
        axB.vlines(i, -0.12, -0.07, color=SPECIES_COLORS.get(species, "#777777"), linewidth=1)
    axB.set_ylim(-0.16, 1.05)
    axB.set_xlim(0, n_total - 1)
    axB.set_ylabel(f"Calibrated probability", fontsize=10)
    axB.set_xlabel(f"Ranked sequence-level prediction", fontsize=10)
    axB.text(0.98, 0.98, "Known reassortant ranking", transform=axB.transAxes, ha="right", va="top", fontsize=10, color="#555555")
    axB.text(0.98, -0.064, "Species", transform=axB.get_yaxis_transform(), ha="right", va="bottom", fontsize=10, color="#555555")

    # Use the calibrated probability scale and choose the operating point by balanced accuracy.
    threshold_grid = np.linspace(0.0, 1.0, 1001)
    scored_thresholds = []
    for threshold in threshold_grid:
        calls = (repeat_plot["probability"].to_numpy() >= threshold).astype(int)
        scored_thresholds.append((balanced_accuracy_score(repeat_plot["target"].to_numpy(), calls), threshold))
    best_t = max(scored_thresholds, key=lambda item: item[0])[1]
    repeat_plot["predicted"] = (repeat_plot["probability"] >= best_t).astype(int)
    actual_yes = int((repeat_plot["target"] == 1).sum())
    actual_no = int((repeat_plot["target"] == 0).sum())
    cm_cells = {
        (0, 0): int(((repeat_plot["predicted"] == 0) & (repeat_plot["target"] == 1)).sum()),
        (0, 1): int(((repeat_plot["predicted"] == 0) & (repeat_plot["target"] == 0)).sum()),
        (1, 0): int(((repeat_plot["predicted"] == 1) & (repeat_plot["target"] == 1)).sum()),
        (1, 1): int(((repeat_plot["predicted"] == 1) & (repeat_plot["target"] == 0)).sum()),
    }
    cm_mat = np.array([
        [cm_cells[(0, 0)] / actual_yes if actual_yes else 0.0, cm_cells[(0, 1)] / actual_no if actual_no else 0.0],
        [cm_cells[(1, 0)] / actual_yes if actual_yes else 0.0, cm_cells[(1, 1)] / actual_no if actual_no else 0.0],
    ])
    axC.imshow(cm_mat, cmap=mpl.cm.Greens, vmin=0, vmax=1)
    for spine in axC.spines.values():
        spine.set_linewidth(1.1)
        spine.set_color("#4a4a4a")
    axC.set_xticks([0, 1], ["Yes", "No"])
    axC.set_yticks([0, 1], ["No", "Yes"])
    axC.set_xlabel(f"Known reassortant", fontsize=10)
    axC.set_ylabel("Predicted reassortant", fontsize=10)
    axC.set_xticks(np.arange(-0.5, 2, 1), minor=True)
    axC.set_yticks(np.arange(-0.5, 2, 1), minor=True)
    axC.grid(which="minor", color="#c7c7c7", linestyle="-", linewidth=0.8)
    axC.tick_params(which="minor", bottom=False, left=False)
    for (r, c), count in cm_cells.items():
        total = actual_yes if c == 0 else actual_no
        txt_color = "white" if cm_mat[r, c] > 0.5 else "#4a4a4a"
        axC.text(c, r - 0.03, f"{count}/{total}", ha="center", va="center", fontsize=14, color=txt_color, fontweight=500)
        axC.text(c, r + 0.18, f"{count / total:.1%}", ha="center", va="center", fontsize=8.0, color=txt_color)

    species_df = (
        repeat_plot.groupby("species", as_index=False)
        .agg(observed_rate=("target", "mean"), predicted_rate=("probability", "mean"), n=("target", "size"), probability_sd=("probability", "std"))
        .sort_values("observed_rate")
    )
    species_df["probability_se"] = species_df["probability_sd"].fillna(0) / np.sqrt(species_df["n"].clip(lower=1))
    envelope_x = np.linspace(0, 1, 300)
    envelope_n = max(1, int(np.median(species_df["n"])))
    envelope_width = 1.96 * np.sqrt(envelope_x * (1 - envelope_x) / envelope_n)
    axD.fill_between(envelope_x, np.clip(envelope_x - envelope_width, 0, 1), np.clip(envelope_x + envelope_width, 0, 1), color="#d7e8f4", alpha=0.56, linewidth=0, zorder=0)
    axD.plot([0, 1], [0, 1], linestyle="--", color="#555555", linewidth=1.0)
    texts = []
    for _, row in species_df.iterrows():
        species_color = SPECIES_COLORS.get(row["species"], "#6c757d")
        yerr = 1.96 * row["probability_se"]
        axD.errorbar(
            row["observed_rate"],
            row["predicted_rate"],
            yerr=[[min(yerr, row["predicted_rate"])], [min(yerr, 1 - row["predicted_rate"])]],
            fmt="o",
            color=species_color,
            ecolor=lighten_color(species_color, 0.55),
            elinewidth=1.0,
            markeredgecolor="white",
            markeredgewidth=0.5,
            markersize=5.4,
            capsize=2.5,    
            zorder=3,
        )
        texts.append(axD.text(row["observed_rate"], row["predicted_rate"], species_short(row["species"]), fontsize=6, color=species_color))
    axD.set_xlim(-0.02, 1.0)
    axD.set_ylim(-0.02, 1.0)
    axD.set_xlabel("Observed reassortment frequency", fontsize=10)
    axD.set_ylabel(f"Predicted reassortment frequency", fontsize=10)
    adjust_text(
        texts,
        x=species_df["observed_rate"].to_numpy(),
        y=species_df["predicted_rate"].to_numpy(),
        ax=axD,
        avoid_self=True,
        force_text=(0.8, 0.8),
        force_static=(1.2, 1.2),
        force_pull=(0.15, 0.15),
        expand=(1.15, 1.25),
        arrowprops=dict(arrowstyle="-", color="#888888", lw=0.5),
    )

    metrics_path = outpath.with_suffix(".calibration.tsv")
    pd.DataFrame([
        {
            "model": focal_model,
            "calibration": calibration_label,
            "n_rows": n_total,
            "positives": int(y_oof.sum()),
            "roc_auc_raw": roc_auc_score(y_oof, raw_prob),
            "average_precision_raw": average_precision_score(y_oof, raw_prob),
            "brier_raw": brier_score_loss(y_oof, raw_prob),
            "roc_auc_calibrated": roc_auc_score(y_oof, calibrated_prob),
            "average_precision_calibrated": average_precision_score(y_oof, calibrated_prob),
            "brier_calibrated": brier_score_loss(y_oof, calibrated_prob),
            "threshold": best_t,
            "balanced_accuracy_at_threshold": balanced_accuracy_score(repeat_plot["target"], repeat_plot["predicted"]),
            "predicted_positive": int(repeat_plot["predicted"].sum()),
        }
    ]).to_csv(metrics_path, sep="\t", index=False)

    fig.savefig(outpath, dpi=600, bbox_inches="tight")
    fig.savefig(outpath.with_suffix(".pdf"), bbox_inches="tight")
    plt.close(fig)
    return outpath


In [12]:
fig3 = make_figure3()
print(fig3)


/Users/ricardorivero/Documents/GitHub/HantaReassortment/notebooks/figures/figure3_model_performance.jpeg
